# Synthetic temporal prediction baselines

Run `python -m reddit_sna.pipelines.prediction --config configs/prediction.toml --run-id week1-prediction-demo` first. Select another run using `REDDIT_PREDICTION_RUN` (relative to the repository root). These synthetic results validate the pipeline; they do not measure Reddit forecasting performance. Metrics, complete prediction populations and confusion matrices are checked before plotting.

## Environment

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, HTML, JSON
import matplotlib.pyplot as plt
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
ROOT = Path(os.environ.get("REDDIT_SNA_PROJECT", Path.cwd())).resolve()
if "REDDIT_SNA_PROJECT" not in os.environ:
    ROOT = next((p for p in [ROOT, *ROOT.parents] if (p / "pyproject.toml").is_file()), ROOT)
if not (ROOT / "pyproject.toml").is_file():
    raise ValueError("Set REDDIT_SNA_PROJECT to the repository root")
print("Active interpreter:", sys.executable)
print("Repository:", ROOT)

from reddit_sna.reporting.prediction import read_prediction, build_prediction_figures

## Read verified saved outputs

In [ ]:
RUN = Path(os.environ.get("REDDIT_PREDICTION_RUN", "artifacts/prediction/week1-prediction-demo"))
if not RUN.is_absolute():
    RUN = ROOT / RUN
report = read_prediction(RUN)
print("Run:", RUN)
print("Rows including unlabeled final observations:", len(report["samples"]))
display(report["split"].groupby("split").size().rename("samples"))
display(report["metrics"])

## Label support and temporal evaluation

In [ ]:
figures = build_prediction_figures(report)
for figure in figures.values():
    plt.close(figure)
display(figures["support"])
print("Split is by label month; preprocessing and model fitting use training rows only.")

## Baseline comparison: preserve both F1 conventions

In [ ]:
display(figures["baselines"])
display(report["per_class"].query("split == 'test'"))
print("macro_f1 averages present truth classes; macro_f1_six averages all six declared events.")

## Every baseline confusion matrix

In [ ]:
for name, figure in figures.items():
    if name.startswith("confusion_"):
        display(figure)
print("A high score on a synthetic mechanism is not evidence of social ground truth or generalization to Reddit.")

## Actual errors and remaining research obligations

In [ ]:
errors = report["predictions"].query("y_true != y_pred")
display(errors.head(30))
print("All saved prediction errors:", len(errors))
print("Inspect rare-class support and fallback rates. B2/B5/HeTCo, real-data adapters, uncertainty and ablations remain later work.")